In [1]:
import pandas as pd


results_df = pd.DataFrame(
    {
        'Model': [],
        'Accuracy': [],
        'Recall': [],
        'ROC-AUC': [],
        'PR-AUC': [],
    }
).astype(
    {
        'Model': str,
        'Accuracy': float,
        'Recall': float,
        'ROC-AUC': float,
        'PR-AUC': float,
    }
)

In [2]:
import joblib
from sklearn.ensemble import VotingClassifier
from module import (
    evaluate_and_append,
    X,
    y,
    skf
)


rf_pipe   = joblib.load('models/best_rf_rus_pipeline.joblib')
xgb_pipe  = joblib.load('models/best_xgb_rus_pipeline.joblib')
cat_pipe  = joblib.load('models/best_catboost_rus_pipeline.joblib')

voting = VotingClassifier(
    estimators=[
        ('rf',  rf_pipe),
        ('xgb', xgb_pipe),
        ('cat', cat_pipe),
    ],
    voting='soft',
    weights=[0.35, 0.15, 0.50],
    n_jobs=-1
)

results_df = evaluate_and_append(
    model_name='VotingClassifier',
    best_estimator=voting,
    X=X, y=y, cv=skf,
    results_df=results_df
)

print(results_df)

              Model  Accuracy    Recall   ROC-AUC    PR-AUC
0  VotingClassifier  0.645784  0.909091  0.831639  0.156134


In [3]:
import numpy as np
from sklearn.ensemble import VotingClassifier

weights_candidates = [
    [0.40, 0.10, 0.50],
    [0.35, 0.15, 0.50],
    [0.30, 0.20, 0.50],
    [0.35, 0.10, 0.55],
    [0.30, 0.10, 0.60],
    [0.25, 0.15, 0.60],
    [0.40, 0.00, 0.60],
    [0.30, 0.00, 0.70],
    [0.20, 0.00, 0.80],
]

best_pr_auc = -1
best_weights = None
best_model_name = None

for w in weights_candidates:
    name = f"Voting_w[{w[0]:.2f}_{w[1]:.2f}_{w[2]:.2f}]"
    print(f"→ {name}")

    voting = VotingClassifier(
        estimators=[('rf', rf_pipe), ('xgb', xgb_pipe), ('cat', cat_pipe)],
        voting='soft',
        weights=w,
        n_jobs=-1
    )

    temp_df = evaluate_and_append(
        model_name=name,
        best_estimator=voting,
        X=X, y=y, cv=skf,
        results_df=results_df
    )

    current_pr_auc = temp_df['PR-AUC'].iloc[-1]
    print(f"PR-AUC = {current_pr_auc:.4f}")

    if current_pr_auc > best_pr_auc:
        best_pr_auc = current_pr_auc
        best_weights = w
        best_model_name = name

print("\n" + "="*60)
print(f"Лучший результат: {best_model_name}")
print(f"Веса: {best_weights}")
print(f"PR-AUC: {best_pr_auc:.4f}")

→ Voting_w[0.40_0.10_0.50]
PR-AUC = 0.1560
→ Voting_w[0.35_0.15_0.50]
PR-AUC = 0.1561
→ Voting_w[0.30_0.20_0.50]
PR-AUC = 0.1565
→ Voting_w[0.35_0.10_0.55]
PR-AUC = 0.1563
→ Voting_w[0.30_0.10_0.60]
PR-AUC = 0.1566
→ Voting_w[0.25_0.15_0.60]
PR-AUC = 0.1569
→ Voting_w[0.40_0.00_0.60]
PR-AUC = 0.1563
→ Voting_w[0.30_0.00_0.70]
PR-AUC = 0.1571
→ Voting_w[0.20_0.00_0.80]
PR-AUC = 0.1580

Лучший результат: Voting_w[0.20_0.00_0.80]
Веса: [0.2, 0.0, 0.8]
PR-AUC: 0.1580


In [5]:
print(temp_df)

                      Model  Accuracy    Recall   ROC-AUC    PR-AUC
0          VotingClassifier  0.645784  0.909091  0.831639  0.156134
1  Voting_w[0.20_0.00_0.80]  0.654398  0.909091  0.830967  0.157980
